In [ ]:
# | default_exp features.wps_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
from kreview.eval_engine import FeatureEvaluator, parse_array

log = structlog.get_logger()

In [ ]:
# | export


class WPSGenomeEvaluator(FeatureEvaluator):
    """Extracts genome-wide WPS metrics with spectral features.

    For each WPS array, extracts:
    - mean, std (original)
    - peak-to-valley amplitude (nucleosome occupancy proxy)
    - median absolute deviation (robust dispersion)
    - spectral max power and dominant frequency (FFT-based periodicity)
    """

    name = "WPSGenome"
    source_file = ".WPS.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "region_type" in cols:
                array_cols = ["wps_nuc", "wps_tf", "prot_frac_nuc", "prot_frac_tf"]
                for row in df.to_dict("records"):
                    rt = (
                        str(row["region_type"])
                        .replace(" ", "_")
                        .replace("-", "_")
                        .replace("|", "_")
                    )
                    for a in array_cols:
                        if a in cols and pd.notna(row[a]):
                            parsed = parse_array(str(row[a]))
                            if len(parsed) > 0:
                                arr = np.array(parsed)
                                extracted[f"{rt}_{a}_mean"] = float(np.mean(arr))
                                extracted[f"{rt}_{a}_std"] = float(np.std(arr))

                                # Peak-to-valley amplitude
                                extracted[f"{rt}_{a}_peak_valley"] = float(
                                    np.max(arr) - np.min(arr)
                                )
                                # Median absolute deviation
                                extracted[f"{rt}_{a}_mad"] = float(
                                    np.median(np.abs(arr - np.median(arr)))
                                )

                                # Spectral features (FFT-based periodicity)
                                if len(arr) >= 50:
                                    fft_vals = np.abs(np.fft.rfft(arr - arr.mean()))
                                    freqs = np.fft.rfftfreq(len(arr))
                                    if len(fft_vals) > 2:
                                        extracted[f"{rt}_{a}_spectral_max_power"] = (
                                            float(np.max(fft_vals[1:]))
                                        )
                                        extracted[
                                            f"{rt}_{a}_spectral_dominant_freq"
                                        ] = float(freqs[1:][np.argmax(fft_vals[1:])])

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
from kreview.features.wps_genomewide import _parse_array

evaluator = WPSGenomeEvaluator()
df = pd.DataFrame([{"region_type": "Promo", "wps_nuc": "[1.0 1.0]"}])
res = evaluator.extract(df)
assert res["Promo_wps_nuc_mean"] == 1.0